# Validating Cointegration with Test Data

This notebook will validate the cointegration strategy from `01_spread_exploration.ipynb` with out-of-sample prices (range between January 1, 2024 to December 31, 2025). 

It will check if the cointegrated pairs identified remain cointegrated with out-of-sample prices (later data)

It will reconstruct the spreads using the in-sample hedge ratios and test whether the cointegration holds out-of-sample, before comparing in-sample and out-of-sample spread statistics.

## Imports

In [1]:
import sys
import os
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore # spread analysis

## Load In-Sample Spread Parameters obtained from Spread Exploration

In [2]:
# load the cointegrated pairs and their stats (hedge ratio, intercept, adf stat, p-value)
cointegrated_pairs = pd.read_csv("../../data/processed/cointegrated_pairs.csv")

# load their summary of spread data (half-life, hurst exponent, z-score, adf-stat, p-value, spread mean, spread std)
summary_df = pd.read_csv("../../data/processed/spread_summary.csv", index_col = 0)

# load the prices log
prices_log = pd.read_csv("../../data/processed/prices_log.csv", index_col=0, parse_dates=True)


### Verify data is loaded successfully

In [3]:
display(cointegrated_pairs)

,y,x,hedge_ratio,intercept,adf_stat,p_value,is_cointegrated
0,CVX.N,XOM.N,0.713964,1.761593,-3.462651,0.009000,True
1,KO.N,PEP.O,0.664058,0.674101,-3.313506,0.014285,True
2,GS.N,MS.N,0.800362,2.256544,-3.252029,0.017164,True


In [4]:
display(summary_df)

,half_life,hurst,current_zscore,spread_mean,spread_std,adf_stat,adf_pvalue,is_stationary
pair,,,,,,,,
CVX.N/XOM.N,33.850675,0.336406,0.628101,-4.021514e-15,0.062838,-3.462651,0.009000,True
KO.N/PEP.O,40.887317,0.418551,0.798284,-1.135710e-14,0.049604,-3.313506,0.014285,True
GS.N/MS.N,36.356403,0.397755,0.592609,5.946136e-15,0.051924,-3.252029,0.017164,True


In [5]:
display(prices_log)

,NVDA.O,AMD.O,TSM.N,KO.N,PEP.O,JPM.N,BAC.N,GS.N,MS.N,XOM.N,CVX.N,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N
Date,,,,,,,,,,,,,,,,,
2019-01-02,1.225392,2.935451,3.597860,3.848657,4.693913,4.598246,3.217275,5.147669,3.698830,4.244057,4.706734,4.343240,4.616308,4.910299,3.965260,4.850075,3.713543
2019-01-03,1.163073,2.836150,3.536893,3.842459,4.684536,4.575844,3.201119,5.132912,3.680847,4.228584,4.687395,4.317675,4.578826,4.880830,3.937174,4.834057,3.685167
2019-01-04,1.225172,2.944439,3.554491,3.862202,4.704835,4.612046,3.241811,5.165072,3.720862,4.264790,4.707907,4.366526,4.624286,4.926891,3.987195,4.850701,3.707746
2019-01-07,1.276758,3.023834,3.561898,3.849083,4.696198,4.612741,3.241029,5.170598,3.730741,4.269977,4.720818,4.400302,4.625561,4.927616,3.985199,4.844266,3.713080
2019-01-08,1.251548,3.032546,3.553632,3.860309,4.705739,4.610854,3.239071,5.166898,3.724488,4.277222,4.716443,4.416778,4.632785,4.959553,3.993944,4.867227,3.717696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,3.888345,4.938781,4.636184,4.065945,5.122057,5.120386,3.509454,5.941880,4.526235,4.624090,5.017611,5.033179,5.925805,5.867572,4.952229,5.046388,3.346389
2023-12-26,3.897498,4.965708,4.648708,4.070052,5.129070,5.126283,3.522234,5.944399,4.530878,4.626344,5.026574,5.033114,5.926019,5.871639,4.952441,5.050753,3.346741
2023-12-27,3.900294,4.984086,4.650621,4.072610,5.132263,5.132263,3.521644,5.951892,4.539671,4.621634,5.023288,5.032658,5.924443,5.880058,4.944282,5.052097,3.353756


## Load Out-of-Sample Prices

Only obtain the prices for the cointegrated pairs

In [6]:
tickers_needed = list(set(cointegrated_pairs["y"].tolist() + cointegrated_pairs["x"].tolist()))
print(tickers_needed)

['GS.N', 'CVX.N', 'MS.N', 'PEP.O', 'KO.N', 'XOM.N']


In [7]:
oos_prices_df = get_close_prices(
    tickers=tickers_needed,
    start=TEST_START,
    end=TEST_END,
    interval=INTERVAL
)

len(oos_prices_df)

[cache partial] Fetching 2024-01-01 → 2024-01-01
LSEG session opened.
[cache partial] Skipping (no trading data): LSEG returned no data for ['GS.N', 'CVX.N', 'MS.N', 'PEP.O', 'KO.N', 'XOM.N'] (2024-01-01 to 2024-01-01)


502

In [8]:
oos_prices_df

,KO.N,PEP.O,CVX.N,MS.N,GS.N,XOM.N
Date,,,,,,
2024-01-02,59.82,172.91,149.48,93.90,388.30,102.36
2024-01-03,59.96,172.95,152.33,91.91,381.79,103.22
2024-01-04,59.76,171.47,150.66,92.15,382.95,102.32
2024-01-05,59.67,168.94,150.40,93.24,386.44,102.63
2024-01-08,60.11,169.11,149.50,93.51,388.86,100.92
...,...,...,...,...,...,...
2025-12-24,70.11,143.74,150.50,181.65,910.78,119.22
2025-12-26,69.87,143.78,150.02,181.87,907.04,119.11
2025-12-29,70.16,144.24,150.99,179.94,892.18,120.53


## Spread Reconstruction with IN-SAMPLE hedge ratios

Convert out-of-sample prices to log prices

In [9]:
oos_prices_log = np.log(oos_prices_df)

Use the estimated hedge ratios and intercept and $\sigma$-bands (mean and standard deviation of spread) from in-sample price series data, to reconstruct the spread for out-of-sample price data

In [10]:
oos_spread = {}

for _, row in cointegrated_pairs.iterrows():
    y_ticker = row["y"]
    x_ticker = row["x"]

    # Ensure both tickers are present in OOS prices
    if y_ticker not in oos_prices_df.columns or x_ticker not in oos_prices_df.columns:
        print(f"Warning: {y_ticker}/{x_ticker} missing from OOS prices, skipping.")
        continue

    spread_oos = compute_spread(
        y           = oos_prices_log[y_ticker],
        x           = oos_prices_log[x_ticker],
        hedge_ratio = row["hedge_ratio"],   # in-sample estimate
        intercept   = row["intercept"],     # in-sample estimate
    )

    oos_spread[f"{y_ticker}/{x_ticker}"] = spread_oos

print(f"Reconstructed {len(oos_spread)} OOS spreads:")
for pair_name, spread in oos_spread.items():
    print(f"  {pair_name}: {spread.first_valid_index().date()} → {spread.last_valid_index().date()}, "
          f"NaNs: {spread.isna().sum()}")


Reconstructed 3 OOS spreads:
  CVX.N/XOM.N: 2024-01-02 → 2025-12-31, NaNs: 0
  KO.N/PEP.O: 2024-01-02 → 2025-12-31, NaNs: 0
  GS.N/MS.N: 2024-01-02 → 2025-12-31, NaNs: 0


Perform sanity check comparing in-sample vs out-of-sample spread statistics.

In [11]:
rows = []
for _, row in cointegrated_pairs.iterrows():
    pair_name = f"{row['y']}/{row['x']}"
    if pair_name not in oos_spread:
        continue

    spread_is  = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    )
    spread_oos = oos_spread[pair_name]

    # is - in sample (2019-2023), oos - out of sample (2024-2025)
    rows.append({
        "pair"          : pair_name,
        "is_mean"       : spread_is.mean(),
        "oos_mean"      : spread_oos.mean(),
        "is_std"        : spread_is.std(),
        "oos_std"       : spread_oos.std(),
        "is_min"        : spread_is.min(),
        "oos_min"       : spread_oos.min(),
        "is_max"        : spread_is.max(),
        "oos_max"       : spread_oos.max(),
    })

sanity_df = pd.DataFrame(rows).set_index("pair")

display(sanity_df)

,is_mean,oos_mean,is_std,oos_std,is_min,oos_min,is_max,oos_max
pair,,,,,,,,
CVX.N/XOM.N,-4.018690e-15,-0.108099,0.062838,0.042914,-0.252323,-0.205582,0.209530,-0.031934
KO.N/PEP.O,-1.135357e-14,0.170275,0.049604,0.111531,-0.168917,-0.017367,0.113686,0.371722
GS.N/MS.N,5.948960e-15,0.247440,0.051924,0.077931,-0.129514,0.069808,0.106505,0.400501


## Testing if Cointegration holds for OOS